# Dependency Matrix Setup
Build out table structures to handle dependency states for processes.

See [the overview](https://github.com/FD-Corporate-IT-Systems/databricks-entdatalakehouse/wiki/dependency-management)

+ process
+ execution_state
+ process_execution_state
+ process_dependency

In [0]:
# %restart_python
import pyspark.sql.types
dbutils.widgets.text("Reset", "false", "reset")
dbutils.widgets.text("Environment", "dev", "environment")

In [0]:
reset = dbutils.widgets.get("Reset")
env = dbutils.widgets.get("Environment").lower()

if env == 'sbx':
    spark.sql("use catalog hive_metastore")
else:
    spark.sql(f"use catalog metadata_{env}")

spark.sql("create schema if not exists control")
spark.sql("use schema control")

tables = [
    "process",
    "execution_state",
    "process_execution_state",
    "process_dependency"
    ]

if reset.lower() in ["true", "1"]:
    print(f"Reset is {reset} - Dropping tables ")
    for t in tables:
        spark.sql(f"DROP TABLE IF EXISTS {t}")

In [0]:
%sql
create table if not exists execution_state (
  id bigint generated always as identity(start with 1 increment by 1) comment "Auto-incrementing record identifier",
  name string comment "Name of the process",
  description string comment "Description of the process"
) 
using delta
comment "Various states of execution for a process."

In [0]:
%sql
create table if not exists process (
  id bigint generated always as identity(start with 1 increment by 1) comment "Auto-incrementing record identifier",
  name string comment "Name of the process",
  description string comment "Description of the process"
) 
using delta
comment "Processes encapsulating some unit of work - truncate and reload, or incremental load of a table, data cleansing, etc."

In [0]:
%sql
create table if not exists process_dependency (
  process_id int comment "Identifier of the process which depends on one or more antecedents.",
  group_id int default 0 comment "Identifier of the dependency group; all conditions in a group must evaluate to true for the group to be true (AND) but the overall result is true if any of the groups evaluate to true (OR).",
  antecedent_id int comment "Identifier of an antecedent process; there can be multiple antecedents.",
  state_ids array<int> comment "List of acceptable execution states for the antecedent process which allow the dependent process to start."
)
using delta
TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported')
comment "Defines the dependencies between a process and its antecedent(s).  Conditions with the same group_id are evaluated with a logical AND while those in different groups are evaluated with a logical OR."


In [0]:
%sql
create table if not exists process_execution_state (
  process_id int comment "Process identifier",
  state_id int comment "Execution state identifier",
  state_change_time timestamp comment "Time of the process entering the given state.",
  target_tables array<string> comment "Array of table(s) affected by the process.",
  row_counts array<int> comment "Array of row counts for the table(s) affected by the process - should be in same order as target_table array.",
  error_messages array<string> comment "Optional - array to hold error messages should any be occur.",
  stack_traces array<string> comment "Optional - array to hold stack traces lines should any be proviced."
)
using delta
comment "The execution states of processes over time.  This is used with process_dependency to determine if processes are allowed to execute.";

ALTER TABLE process_execution_state SET TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported');
alter table process_execution_state alter column state_change_time set default (current_timestamp);


# Seed Data

In [0]:
%sql

with s as (
    select 'Queue' as name, 'Process has requested to run and is in a queue waiting.' as description
    union
    select 'Starting', 'Process is starting to execute.'
    union
    select 'Started', 'Process has started execution.'
    union
    select 'Running', 'Process is running.'
    union
    select 'Paused', 'Process has been paused or interrupted in progress.'
    union
    select 'Resumed', 'Process has been resumed from a paused or interrupted state.'
    union
    select 'Succeeded', 'Process has completed successfully.'
    union
    select 'Failed', 'Process has failed.'
    union
    select 'Cancelled', 'Process has been cancelled.'
    union
    select 'Complete', 'Process has finished execution.'
)
insert into execution_state (name, description)
select name, description
from s
where not exists (select name from execution_state where name = s.name)